In [ ]:
!pip install -q tensorflow-datasets tensorflow numpy matplotlib pillow scikit-learn

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.vgg16 import (
    VGG16, preprocess_input, decode_predictions
)

# 1. Load ImageNet-Mini dataset (local folder)
DATASET_DIR = "ImageNet-Mini/images"

dataset = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    labels="inferred",
    label_mode="int",
    image_size=(224, 224),
    batch_size=None,
    shuffle=False
)

names = dataset.class_names

images = []
labels = []

for img, label in dataset:
    images.append(img.numpy())
    labels.append(int(label.numpy()))

images = np.array(images, dtype=np.float32)
labels = np.array(labels, dtype=np.int32)

print("Number of classes:", len(names))
print("Classes:", names)
print("Total images:", len(images))

# 2. Split into training and validation sets
train_idx, val_idx = train_test_split(
    np.arange(len(images)),
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train = tf.data.Dataset.from_tensor_slices(
    (images[train_idx], labels[train_idx])
)

test = tf.data.Dataset.from_tensor_slices(
    (images[val_idx], labels[val_idx])
)

print("Training images:", len(train_idx))
print("Validation images:", len(val_idx))

# 3. Randomly select 5 test images
samples = list(test.shuffle(
    buffer_size=len(val_idx),
    seed=42,
    reshuffle_each_iteration=False
).take(5))

imgs, true_labels = [], []

for img, label in samples:
    imgs.append(img.numpy())
    true_labels.append(names[int(label.numpy())])

imgs = np.array(imgs, dtype=np.float32)

# 4. VGG16 preprocessing
X = preprocess_input(imgs.copy())

print("\nPreprocessing: Resize -> VGG16 preprocess_input")
print("Original images:", imgs.shape)
print("Preprocessed images:", X.shape)

# 5. Load pretrained VGG16
model = VGG16(weights="imagenet", include_top=True) # If internet is there
#model = VGG16(weights="vgg16_weights_tf_dim_ordering_tf_kernels.h5") #If downloaded locally

# 6. Create intermediate feature extraction model
feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("block3_conv3").output
)

# 7. Predict ImageNet classes
pred = model.predict(X, verbose=0)

# 8. Extract intermediate feature maps
features = feature_model.predict(X, verbose=0)

# 9. Display images, original labels and Top-5 predictions
for i in range(len(imgs)):
    top5 = decode_predictions(pred[i:i+1], top=5)[0]

    plt.figure(figsize=(4, 4))
    plt.imshow(np.clip(imgs[i], 0, 255).astype("uint8"))
    plt.title("Original: " + true_labels[i])
    plt.axis("off")
    plt.show()

    print("\nOriginal dataset class:", true_labels[i])
    print("Top-5 ImageNet predictions:")

    for _, name, prob in top5:
        print(f"{name:25s} {prob * 100:.2f}%")

    print("Predicted class:", top5[0][1])

# 10. Visualize one intermediate feature map
plt.figure(figsize=(6, 6))
plt.imshow(features[0, :, :, 0], cmap="viridis")
plt.title("Intermediate Feature Map - block3_conv3")
plt.axis("off")
plt.show()

# 11. Output dimensions
print("\n----- OUTPUT DIMENSIONS -----")
print("Input:", X.shape)
print("Intermediate features:", features.shape)
print("Final prediction:", pred.shape)
